### 03_build_neural_network

##### Purpose

Build the first PyTorch neural network for the Telco Customer Churn project and understand how PyTorch represents neural-network layers, neurons, weights, biases, parameters, forward propagation, logits, probabilities, and predicted classes.

This notebook focuses on building and verifying the neural-network architecture only.

Model training will be performed in the next notebook.

##### Technologies Used

- Python
- PyTorch
- torch.nn
- TensorDataset / DataLoader outputs from Notebook 02
- Databricks

##### Input

Prepared PyTorch tensors from 02_data_preparation:

X_train_tensor → [4930, 45]

y_train_tensor → [4930, 1]

X_val_tensor   → [1056, 45]

y_val_tensor   → [1056, 1]

X_test_tensor  → [1057, 45]

y_test_tensor  → [1057, 1]

The preprocessing pipeline converted the original Telco features into:

45 model-ready input features

through:

``` text 

Numerical features
    ↓
Imputation + Standardization

Categorical features
    ↓
Imputation + One-Hot Encoding

Binary feature
    ↓
Passthrough

```

Architecture

The neural network used in this project is:

``` text 

45 Input Features
       ↓
Linear Layer
45 → 32
       ↓
ReLU
       ↓
Linear Layer
32 → 16
       ↓
ReLU
       ↓
Output Layer
16 → 1
       ↓
Raw Logit

```

Architecture shorthand:

45 → 32 → 16 → 1

Where:

45
→ determined by preprocessing

32
→ hidden-layer hyperparameter

16
→ hidden-layer hyperparameter

1
→ one output for binary classification

##### 1. Import PyTorch Neural-Network Module

In [0]:
import torch
import torch.nn as nn

##### 2. Determine Input Size

In [0]:
input_size = X_train_tensor.shape[1]

print("Input size:", input_size)

##### 3. Define the Neural Network

In [0]:
class TelcoChurnNN(nn.Module):

    def __init__(self, input_size):
        super().__init__()

        self.layer1 = nn.Linear(
            input_size,
            32,
        )

        self.relu1 = nn.ReLU()

        self.layer2 = nn.Linear(
            32,
            16,
        )

        self.relu2 = nn.ReLU()

        self.output_layer = nn.Linear(
            16,
            1,
        )

    def forward(self, x):

        x = self.layer1(x)
        x = self.relu1(x)

        x = self.layer2(x)
        x = self.relu2(x)

        x = self.output_layer(x)

        return x

##### 4. Understanding nn.Module

In [0]:
class TelcoChurnNN(nn.Module):

means:

TelcoChurnNN is our neural-network class and inherits PyTorch neural-network functionality from nn.Module.

nn.Module provides functionality used later such as:

model.parameters()
model.named_parameters()
model.train()
model.eval()

##### 5. Understanding __init__()

In [0]:
def __init__(self, input_size):

defines the components contained in the model.

Conceptually:

__init__()

Defines:

Linear(45,32)
ReLU

Linear(32,16)
ReLU

Linear(16,1)

The constructor runs when the model object is created.

##### 6. Understanding super().__init__()

In [0]:
super().__init__()

initializes the functionality inherited from:

nn.Module

This allows PyTorch to correctly register and manage the model's layers and trainable parameters.

##### 7. Understanding forward()

In [0]:
def forward(self, x):

defines how data travels through the network.

``` text 

Input
 ↓
Layer 1
 ↓
ReLU
 ↓
Layer 2
 ↓
ReLU
 ↓
Output Layer
 ↓
Return Logit

```

This is the implementation of forward propagation.

A useful mental model is:

__init__()

What components does the network contain?


forward()

How does data move through those components?

##### 8. Create the Model

In [0]:
model = TelcoChurnNN(
    input_size=input_size
)

Creating the model causes PyTorch to initialize the layers, weights, and biases.

At this point:

Weights exist
Biases exist

BUT

Training has not happened.

##### 9. Inspect the Model Architecture

In [0]:
print(model)

##### 10. Understanding nn.Linear

In [0]:
nn.Linear(
    in_features,
    out_features
)

For example:

nn.Linear(45, 32)

means:

45 incoming values
       ↓
32 neurons
       ↓
32 output values

Each of the 32 neurons receives all 45 incoming features.

##### 11. Inspect Layer 1

In [0]:
print(model.layer1)

Expected:

Linear(
    in_features=45,
    out_features=32,
    bias=True
)

Layer 1 contains:

45×32=1440

weights and:

32

biases.

Total:

1440+32=1472

trainable parameters.

##### 12. Inspect Layer 1 Weights

In [0]:
print(
    model.layer1.weight
)

In [0]:
print(
    "Layer 1 weight shape:",
    model.layer1.weight.shape
)

Expected:

torch.Size([32, 45])

Meaning:

32 neurons
×
45 incoming weights per neuron

##### 13. Inspect One Neuron

In [0]:
print(
    model.layer1.weight[0]
)

In [0]:
print(
    "Weights for neuron 1:",
    model.layer1.weight[0].shape
)

##### 14. Inspect Biases

In [0]:
print(
    model.layer1.bias
)

print(
    "Layer 1 bias shape:",
    model.layer1.bias.shape
)

print(
    "Neuron 1 bias:",
    model.layer1.bias[0]
)

##### 15. Inspect Remaining Layer Shapes

In [0]:
print(
    "Layer 2 weight shape:",
    model.layer2.weight.shape
)

print(
    "Layer 2 bias shape:",
    model.layer2.bias.shape
)

print(
    "Output weight shape:",
    model.output_layer.weight.shape
)

print(
    "Output bias shape:",
    model.output_layer.bias.shape
)

##### 16. Count Trainable Parameters

In [0]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(
    "Total parameters:",
    total_parameters
)

In [0]:
#manual calculation:

Layer 1

45 × 32 = 1440 weights
32 biases

Total = 1472


Layer 2

32 × 16 = 512 weights
16 biases

Total = 528


Output Layer

16 × 1 = 16 weights
1 bias

Total = 17

1472+528+17=2017​

##### 17. Inspect Parameters by Name

In [0]:
for name, parameter in model.named_parameters():

    print(
        name,
        parameter.shape,
        parameter.numel(),
    )

Expected:

layer1.weight        [32,45]   1440
layer1.bias          [32]        32

layer2.weight        [16,32]    512
layer2.bias          [16]        16

output_layer.weight  [1,16]      16
output_layer.bias    [1]          1

.numel() means:

number of elements contained in a tensor.

##### 18. Understanding requires_grad=True

When inspecting trainable parameters, PyTorch displays:

requires_grad=True

This means:

PyTorch tracks operations involving this parameter so that its gradient can later be calculated during backpropagation.

Training will eventually perform:

``` text 

Forward Propagation
       ↓
Loss
       ↓
loss.backward()
       ↓
Gradients
       ↓
Optimizer
       ↓
Updated Weights + Biases

```

##### 19. Verify One Customer

In [0]:
one_customer = X_train_tensor[0].unsqueeze(0)

print(
    "One customer shape:",
    one_customer.shape,
)

##### 20. Run Forward Propagation

In [0]:
logit = model(
    one_customer
)

print(
    "Raw model output:",
    logit,
)

print(
    "Output shape:",
    logit.shape,
)

##### 21. Verify Intermediate Shapes

In [0]:
x = one_customer

print(
    "Input:",
    x.shape,
)

x = model.layer1(x)

print(
    "After Layer 1:",
    x.shape,
)

x = model.relu1(x)

print(
    "After ReLU 1:",
    x.shape,
)

x = model.layer2(x)

print(
    "After Layer 2:",
    x.shape,
)

x = model.relu2(x)

print(
    "After ReLU 2:",
    x.shape,
)

x = model.output_layer(x)

print(
    "After Output Layer:",
    x.shape,
)

##### 22. Inspect ReLU

In [0]:
layer1_output = model.layer1(
    one_customer
)

print(
    "Before ReLU:"
)

print(
    layer1_output
)


relu1_output = model.relu1(
    layer1_output
)

print(
    "After ReLU:"
)

print(
    relu1_output
)

##### 23. Convert Logit to Probability

In [0]:
probability = torch.sigmoid(
    logit
)

print(
    "Churn probability:",
    probability,
)

##### 24. Convert Probability to Class

In [0]:
predicted_class = (
    probability >= 0.5
).float()

print(
    "Predicted class:",
    predicted_class,
)

In [0]:
print(
    "Probability:",
    probability.item(),
)

print(
    "Predicted class:",
    int(predicted_class.item()),
)

print(
    "Actual class:",
    int(actual.item()),
)

Remember that the model is still untrained, so this correct prediction is coincidental and is not evidence of model performance.

##### 25. Verify a Complete Batch

In [0]:
batch_logits = model(
    X_batch
)

print(
    "Input batch shape:",
    X_batch.shape,
)

print(
    "Output logits shape:",
    batch_logits.shape,
)

##### 26. Convert Batch Logits to Probabilities

In [0]:
batch_probabilities = torch.sigmoid(
    batch_logits
)

print(
    batch_probabilities
)

##### 27. Convert Batch Probabilities to Classes

In [0]:
batch_predictions = (
    batch_probabilities >= 0.5
).float()

print(
    batch_predictions
)

##### 28. Prediction Without Gradient Tracking

In [0]:
with torch.no_grad():

    batch_logits = model(
        X_batch
    )

    batch_probabilities = torch.sigmoid(
        batch_logits
    )

    batch_predictions = (
        batch_probabilities >= 0.5
    ).float()

##### 29. Final Shape Verification

In [0]:
29. Final Shape Verificationprint(
    "X_batch:",
    X_batch.shape,
)

print(
    "Logits:",
    batch_logits.shape,
)

print(
    "Probabilities:",
    batch_probabilities.shape,
)

print(
    "Predictions:",
    batch_predictions.shape,
)

print(
    "Actual targets:",
    y_batch.shape,
)

##### Key Learnings


- nn.Module : Base PyTorch class used to create neural-network models.

- nn.Linear : Creates a fully connected linear layer:

        ``` text 

        input
        ↓
        weighted sum + bias
        ↓
        output

        ```

- Hidden neurons are hyperparameters

    - architecture:45 → 32 → 16 → 1

    - contains:

        - 45 → determined by input data

        - 32 → chosen hyperparameter

        - 16 → chosen hyperparameter

        - 1 → binary classification output

- Every neuron receives all previous-layer outputs

    - For: nn.Linear(45, 32).    each of the 32 neurons receives all 45 input features.

-  Weights and biases are trainable parameters

    - Our network contains: 2017 trainable parameters

-  ReLU introduces non-linearity

    - negative → 0
    - positive → unchanged

-  Forward propagation

        Data moves:

        ``` text

        Input
        ↓
        Hidden Layers
        ↓
        Output

        ```

-  Logit

    - The output layer produces a raw score called a logit.

-  Sigmoid

        Converts:

        ``` text 

        Logit
        ↓
        Probability between 0 and 1

        ```

-  Classification threshold

        Converts:

        ``` text 

        Probability
        ↓
        Class 0 or 1

        ```

-  The model has not learned yet

    - Creating the model initializes parameters.

    - Learning begins only when training performs:

        ``` text 

        Forward Pass
        ↓
        Loss
        ↓
        Backpropagation
        ↓
        Gradients
        ↓
        Optimizer
        ↓
        Parameter Updates

        ```

##### Conclusion


In this notebook, we built the first PyTorch neural network for the Telco Customer Churn project.

The network architecture is:

``` text 

45 Input Features
       ↓
32 Hidden Neurons
       ↓
ReLU
       ↓
16 Hidden Neurons
       ↓
ReLU
       ↓
1 Output Neuron
       ↓
Logit

```

We inspected the actual weights and biases created by PyTorch, verified the network's 2,017 trainable parameters, followed a real customer through the network, and converted the final logit into a churn probability and predicted class.

The model architecture is now ready for training.

##### Next Notebook

04_train_neural_network

The next notebook will introduce the actual learning process:

``` text 

Training Batch
     ↓
Forward Propagation
     ↓
Logits
     ↓
BCEWithLogitsLoss
     ↓
Loss
     ↓
Backpropagation
     ↓
Gradients
     ↓
Adam Optimizer
     ↓
Update Weights + Biases
     ↓
Repeat Across Epochs

```

This is where the randomly initialized neural network finally begins learning patterns associated with customer churn.